# RT-DETR-X Crack Training (Kaggle)

This notebook is built for Kaggle `Run All`.

What it does:
- auto-detects the crack dataset under `/kaggle/input`
- sanitizes labels into a clean runtime dataset
- trains `rtdetr-x.pt` as the heavy RT-DETR baseline
- uses conservative batch fallback to avoid OOM on T4/T4x2
- runs a short fine-tune phase for extra mAP
- evaluates on `val` and `test`
- exports a compact training bundle at the end


In [ ]:
!pip -q install -U "ultralytics>=8.4.20" "pyyaml>=6.0" "opencv-python-headless>=4.8.0"

import csv
import gc
import json
import os
import random
import shutil
import time
import zipfile
from pathlib import Path

import cv2
import numpy as np
import torch
import yaml
from ultralytics import RTDETR

print('RTDETR:', RTDETR)
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU{i}:', torch.cuda.get_device_name(i))


In [ ]:
# ===== Config =====
DATASET_HINT = 'crack-dataset-yolo26'   # substring match; set '' for auto
MODEL_NAME = 'rtdetr-x.pt'
RUN_NAME = 'rtdetr_x_1024_crack_v1'
PROJECT_DIR = Path('/kaggle/working/runs/detect')
SANITIZED_ROOT = Path('/kaggle/working/input_dataset_clean_rtdetr')
RUNTIME_YAML = Path('/kaggle/working/data_runtime_rtdetr.yaml')
REPORT_JSON = Path('/kaggle/working/report_metrics_rtdetr.json')
REPORT_CSV = Path('/kaggle/working/report_metrics_rtdetr.csv')
SEED = 42

IMAGE_SIZE = 1024
PHASE1_EPOCHS = 56
PHASE2_EPOCHS = 18
RUN_PHASE2 = True
PATIENCE_PHASE1 = 20
PATIENCE_PHASE2 = 8
SAVE_PERIOD = 10
WORKERS = 4
CACHE = False

BATCH_CANDIDATES_1GPU = [2, 1]
BATCH_CANDIDATES_2GPU = [4, 2, 1]
TRAIN_DEVICE = '0,1' if torch.cuda.device_count() >= 2 else ('0' if torch.cuda.is_available() else 'cpu')
EVAL_DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print('Train device:', TRAIN_DEVICE)
print('Eval device :', EVAL_DEVICE)


In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
torch.backends.cudnn.benchmark = True
if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

INPUT_ROOT = Path('/kaggle/input')
assert INPUT_ROOT.exists(), 'This notebook is intended for Kaggle.'

def read_yaml(path: Path):
    return yaml.safe_load(path.read_text(encoding='utf-8'))

def find_split_dir(base: Path, names):
    for split in names:
        candidate = base / split / 'images'
        if candidate.exists():
            return candidate
    return None

def discover_dataset_yaml(dataset_hint=''):
    cands = []
    for p in INPUT_ROOT.rglob('data.yaml'):
        try:
            data = read_yaml(p)
        except Exception:
            continue
        if not isinstance(data, dict):
            continue
        score = 0
        if dataset_hint and dataset_hint.lower() in str(p).lower():
            score += 10
        if 'train' in data and ('val' in data or 'valid' in str(p).lower()):
            score += 3
        cands.append((score, p))
    assert cands, 'No data.yaml found under /kaggle/input'
    cands.sort(key=lambda x: (x[0], len(str(x[1]))), reverse=True)
    return cands[0][1]

yaml_path = discover_dataset_yaml(DATASET_HINT)
yaml_data = read_yaml(yaml_path)
dataset_root = yaml_path.parent

train_images = find_split_dir(dataset_root, ['train']) or find_split_dir(dataset_root.parent, ['train'])
val_images = find_split_dir(dataset_root, ['valid', 'val']) or find_split_dir(dataset_root.parent, ['valid', 'val'])
test_images = find_split_dir(dataset_root, ['test']) or find_split_dir(dataset_root.parent, ['test'])

assert train_images is not None, f'Train images not found near {yaml_path}'
assert val_images is not None, f'Val images not found near {yaml_path}'
assert test_images is not None, f'Test images not found near {yaml_path}'

split_image_dirs = {
    'train': train_images,
    'val': val_images,
    'test': test_images,
}

def resolve_label_dir(image_dir: Path) -> Path:
    # Roboflow format: split/{images,labels}
    return image_dir.parent / 'labels'

def sanitize_label_file(src: Path, dst: Path):
    lines_out = []
    seen = set()
    stats = {'box_lines': 0, 'segment_lines_converted': 0, 'invalid_lines_dropped': 0, 'duplicate_boxes_removed': 0}
    if src.exists():
        raw_lines = src.read_text(encoding='utf-8', errors='ignore').splitlines()
    else:
        raw_lines = []
    for raw in raw_lines:
        s = raw.strip()
        if not s:
            continue
        parts = s.split()
        try:
            cls_id = int(float(parts[0]))
            nums = [float(x) for x in parts[1:]]
        except Exception:
            stats['invalid_lines_dropped'] += 1
            continue

        if len(nums) == 4:
            xc, yc, w, h = nums
            stats['box_lines'] += 1
        elif len(nums) >= 6 and len(nums) % 2 == 0:
            xs = nums[0::2]
            ys = nums[1::2]
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)
            xc = (x1 + x2) / 2.0
            yc = (y1 + y2) / 2.0
            w = max(0.0, x2 - x1)
            h = max(0.0, y2 - y1)
            stats['segment_lines_converted'] += 1
        else:
            stats['invalid_lines_dropped'] += 1
            continue

        xc = min(1.0, max(0.0, xc))
        yc = min(1.0, max(0.0, yc))
        w = min(1.0, max(0.0, w))
        h = min(1.0, max(0.0, h))
        if w <= 1e-6 or h <= 1e-6:
            stats['invalid_lines_dropped'] += 1
            continue
        row = (cls_id, round(xc, 6), round(yc, 6), round(w, 6), round(h, 6))
        if row in seen:
            stats['duplicate_boxes_removed'] += 1
            continue
        seen.add(row)
        lines_out.append(f'{cls_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')

    dst.write_text('\n'.join(lines_out), encoding='utf-8')
    return stats, len(lines_out)

if SANITIZED_ROOT.exists():
    shutil.rmtree(SANITIZED_ROOT)
SANITIZED_ROOT.mkdir(parents=True, exist_ok=True)

names = yaml_data.get('names', ['crack'])
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]

summary = {}
global_stats = {'box_lines': 0, 'segment_lines_converted': 0, 'invalid_lines_dropped': 0, 'duplicate_boxes_removed': 0}

for split, img_dir in split_image_dirs.items():
    out_img_dir = SANITIZED_ROOT / split / 'images'
    out_lab_dir = SANITIZED_ROOT / split / 'labels'
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lab_dir.mkdir(parents=True, exist_ok=True)
    src_lab_dir = resolve_label_dir(img_dir)

    images = sorted([p for p in img_dir.iterdir() if p.is_file()])
    pos_images = 0
    boxes = 0
    for img_path in images:
        shutil.copy2(img_path, out_img_dir / img_path.name)
        src_label = src_lab_dir / f'{img_path.stem}.txt'
        dst_label = out_lab_dir / f'{img_path.stem}.txt'
        stats, kept = sanitize_label_file(src_label, dst_label)
        for k, v in stats.items():
            global_stats[k] += int(v)
        boxes += kept
        if kept > 0:
            pos_images += 1

    summary[split] = {
        'images': len(images),
        'pos_images': pos_images,
        'neg_images': len(images) - pos_images,
        'boxes': boxes,
    }

runtime_yaml = {
    'path': str(SANITIZED_ROOT),
    'train': str((SANITIZED_ROOT / 'train' / 'images').resolve()),
    'val': str((SANITIZED_ROOT / 'val' / 'images').resolve()),
    'test': str((SANITIZED_ROOT / 'test' / 'images').resolve()),
    'nc': len(names),
    'names': names,
}
RUNTIME_YAML.write_text(yaml.safe_dump(runtime_yaml, sort_keys=False, allow_unicode=False), encoding='utf-8')

print('Using dataset root :', dataset_root)
print('Source data.yaml   :', yaml_path)
print('Runtime data.yaml  :', RUNTIME_YAML)
print('Summary            :', json.dumps(summary, indent=2))
print('Sanitize stats     :', global_stats)

assert summary['train']['images'] > 0, 'Empty train split'
assert summary['val']['images'] > 0, 'Empty val split'
print('PREFLIGHT PASS')


In [ ]:
def best_effort_empty_cache():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_with_batch_fallback(model_source, run_name, train_args, batch_candidates):
    last_error = None
    for batch_size in batch_candidates:
        best_effort_empty_cache()
        print(f'\nStart {run_name} | model={model_source} | batch={batch_size}')
        try:
            model = RTDETR(model_source)
            model.train(name=run_name, batch=batch_size, **train_args)
            return batch_size, None
        except Exception as exc:
            last_error = exc
            msg = str(exc).lower()
            print(f'Attempt failed with batch={batch_size}: {exc}')
            if ('out of memory' in msg) or ('cuda error' in msg) or ('cudnn' in msg):
                continue
            raise
    raise RuntimeError(f'All batch attempts failed. Last error: {last_error}')

batch_candidates = BATCH_CANDIDATES_2GPU if torch.cuda.device_count() >= 2 else BATCH_CANDIDATES_1GPU
print('Batch candidates:', batch_candidates)

phase1_args = dict(
    data=str(RUNTIME_YAML),
    device=TRAIN_DEVICE,
    epochs=PHASE1_EPOCHS,
    imgsz=IMAGE_SIZE,
    optimizer='AdamW',
    lr0=0.00035,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,
    pretrained=True,
    amp=True,
    cache=CACHE,
    workers=WORKERS,
    save=True,
    save_period=SAVE_PERIOD,
    plots=True,
    val=True,
    verbose=True,
    project=str(PROJECT_DIR),
    exist_ok=True,
    seed=SEED,
    deterministic=False,
    patience=PATIENCE_PHASE1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=1.5,
    translate=0.06,
    scale=0.30,
    shear=0.0,
    perspective=0.0005,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.20,
    mixup=0.03,
    copy_paste=0.0,
    close_mosaic=12,
    erasing=0.15,
)

phase2_args = dict(
    data=str(RUNTIME_YAML),
    device=TRAIN_DEVICE,
    epochs=PHASE2_EPOCHS,
    imgsz=IMAGE_SIZE,
    optimizer='AdamW',
    lr0=0.00008,
    lrf=0.05,
    weight_decay=0.0005,
    warmup_epochs=2.0,
    cos_lr=True,
    pretrained=True,
    amp=True,
    cache=CACHE,
    workers=WORKERS,
    save=True,
    save_period=0,
    plots=True,
    val=True,
    verbose=True,
    project=str(PROJECT_DIR),
    exist_ok=True,
    seed=SEED,
    deterministic=False,
    patience=PATIENCE_PHASE2,
    hsv_h=0.010,
    hsv_s=0.35,
    hsv_v=0.2,
    degrees=0.5,
    translate=0.02,
    scale=0.12,
    shear=0.0,
    perspective=0.0,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=1,
    erasing=0.0,
)


In [ ]:
phase1_name = f'{RUN_NAME}_phase1'
phase1_batch, _ = train_with_batch_fallback(MODEL_NAME, phase1_name, phase1_args, batch_candidates)
phase1_dir = PROJECT_DIR / phase1_name
phase1_best = phase1_dir / 'weights' / 'best.pt'
phase1_last = phase1_dir / 'weights' / 'last.pt'
assert phase1_best.exists(), f'Missing phase1 best: {phase1_best}'
print('phase1_batch_used:', phase1_batch)
print('phase1_best      :', phase1_best)
print('phase1_last      :', phase1_last)


In [ ]:
final_dir = phase1_dir
final_best = phase1_best
final_last = phase1_last
phase2_batch = None

if RUN_PHASE2:
    phase2_name = f'{RUN_NAME}_phase2'
    phase2_batch, _ = train_with_batch_fallback(str(phase1_best), phase2_name, phase2_args, batch_candidates)
    final_dir = PROJECT_DIR / phase2_name
    final_best = final_dir / 'weights' / 'best.pt'
    final_last = final_dir / 'weights' / 'last.pt'
    assert final_best.exists(), f'Missing phase2 best: {final_best}'

print('phase2_batch_used:', phase2_batch)
print('final_dir        :', final_dir)
print('final_best       :', final_best)
print('final_last       :', final_last)


In [ ]:
def pack_metrics(metrics):
    return {
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    }

best_effort_empty_cache()
model = RTDETR(str(final_best))
eval_batch = 1 if IMAGE_SIZE >= 1024 else 2

val_metrics = model.val(data=str(RUNTIME_YAML), split='val', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=eval_batch, plots=True, save_json=False, verbose=True)
test_metrics = model.val(data=str(RUNTIME_YAML), split='test', device=EVAL_DEVICE, imgsz=IMAGE_SIZE, batch=eval_batch, plots=False, save_json=False, verbose=True)

report = {
    'model_name': MODEL_NAME,
    'run_name': RUN_NAME,
    'image_size': IMAGE_SIZE,
    'train_device': TRAIN_DEVICE,
    'phase1_batch': phase1_batch,
    'phase2_batch': phase2_batch,
    'phase1_best': str(phase1_best),
    'final_best': str(final_best),
    'val': pack_metrics(val_metrics),
    'test': pack_metrics(test_metrics),
    'dataset_summary': summary,
}

REPORT_JSON.write_text(json.dumps(report, indent=2), encoding='utf-8')
with REPORT_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['split', 'mAP50', 'mAP50-95', 'precision', 'recall'])
    writer.writeheader()
    writer.writerow({'split': 'val', **report['val']})
    writer.writerow({'split': 'test', **report['test']})

print(json.dumps(report, indent=2))


In [ ]:
bundle_path = Path('/kaggle/working') / f'{RUN_NAME}_bundle.zip'
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in [RUNTIME_YAML, REPORT_JSON, REPORT_CSV]:
        if p.exists():
            zf.write(p, arcname=p.name)
    for p in [final_best, final_last, phase1_best, phase1_last]:
        if p.exists():
            zf.write(p, arcname=f'weights/{p.name}')
    for rel in ['args.yaml', 'results.csv', 'results.png', 'confusion_matrix.png', 'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png']:
        p = final_dir / rel
        if p.exists():
            zf.write(p, arcname=f'final_run/{rel}')
    if phase1_dir != final_dir:
        p = phase1_dir / 'results.csv'
        if p.exists():
            zf.write(p, arcname='phase1/results.csv')

print('Bundle saved:', bundle_path)
print('Bundle size MB:', round(bundle_path.stat().st_size / (1024 * 1024), 2))
print('Kaggle output files ready.')
